# Processing stackSentinel over CA

In [ ]:
import os
#os.environ["OMP_NUM_THREADS"] = "4" # export OMP_NUM_THREADS=4
#os.environ["OPENBLAS_NUM_THREADS"] = "1" # export OPENBLAS_NUM_THREADS=4 
#os.environ["MKL_NUM_THREADS"] = "6" # export MKL_NUM_THREADS=6
#os.environ["VECLIB_MAXIMUM_THREADS"] = "4" # export VECLIB_MAXIMUM_THREADS=4
#os.environ["NUMEXPR_NUM_THREADS"] = "6" # export NUMEXPR_NUM_THREADS=6

In [ ]:
import site
from pathlib import Path
import subprocess
import numpy as np
import time
import zipfile

import requests
from lxml import etree
import urllib.request
from urllib.parse import urljoin

# import rasterio
# from rasterio import logging
# log = logging.getLogger()
# log.setLevel(logging.ERROR)

import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from shapely import Polygon

# Plotting modules
from IPython import display
from cartopy import crs as ccrs
from matplotlib import pyplot as plt
import cartopy.io.img_tiles as cimgt

# isce2
import isce
isce_application_path = Path(isce.isce_path) / 'applications'
os.environ['PATH'] += (':' + str(isce_application_path))

# TopsStack aux modules
import asf_search as asf
import eof 
# from dem_stitcher.stitcher import stitch_dem

# Adjustable parameters

In [ ]:
# Work Directory and Area of Interest
work_dir = Path('/mnt/hgfs/Strangnas/asc_slc_2')
aoi = [59.30, 59.596471, 16.00, 17.161318] # snwe 

# Download parameters
# track = 144
# start_date = '2023-02-01'
# end_date = '2023-04-10'
# flight_direction = 'DSC'
# n_processes = 50 # number of threads for downloading SLCs
# dem_name = 'glo_30'

In [ ]:
## Set-up processing directory structure

# Sentinel-1 SLCs folder
slc_dir = work_dir / 'slc'
slc_dir.mkdir(exist_ok=True, parents=True)

# GLO-30 DEM folder
dem_dir = work_dir / 'dem'
dem_dir.mkdir(exist_ok=True, parents=True)
dem_path = dem_dir / 'full_res.dem.wgs84'

# Sentinel-1 ORBIT data folder
orbit_dir = work_dir / 'orbits'
orbit_dir.mkdir(exist_ok=True, parents=True)

# Sentinel-1 AUX CAL-file folder
aux_dir = work_dir / 'AUX'
aux_dir.mkdir(exist_ok=True, parents=True)

# stackSentinel dir
isce_run_dir = slc_dir.parent / 'isce'
isce_run_dir.mkdir(exist_ok=True, parents=True)

run_dir = isce_run_dir / 'run_files'
run_ifg_dir = isce_run_dir / 'run_ifg_files'

# Stack Sentinel - make coregistrated SLC stack

In [ ]:
# We need to add isce2/contrib/stack/ directory to our PATH env variable
# Use dynamic path resolution based on the ISCE installation
import isce
isce_path = os.path.dirname(os.path.dirname(isce.__file__))
isce2_stack_dir = Path(isce_path) / 'contrib' / 'stack'

# Alternative paths to check for stackSentinel.py
possible_paths = [
    isce2_stack_dir / 'topsStack',
    Path('/home/roy/miniconda3/envs/insar/share/isce2/topsStack'),
    Path('/home/roy/tools/isce2/src/isce2/contrib/stack/topsStack'),
    Path(isce_path) / 'share' / 'isce2' / 'topsStack'
]

# Find the correct path that contains stackSentinel.py
stack_path = None
for path in possible_paths:
    if (path / 'stackSentinel.py').exists():
        stack_path = path
        break

if stack_path:
    print(f'Found stackSentinel.py in: {stack_path}')
    os.environ['PATH'] = str(stack_path) + ':' + os.environ.get('PATH', '')
    # Set PYTHONPATH to include the contrib/stack directory
    pythonpath_dirs = [
        str(isce2_stack_dir),
        str(stack_path.parent),  # This should be the share/isce2 directory
        str(isce_path)
    ]
    os.environ['PYTHONPATH'] = ':'.join(pythonpath_dirs) + ':' + os.environ.get('PYTHONPATH', '')
    print(f'PYTHONPATH set to: {os.environ["PYTHONPATH"][:200]}...')
else:
    print('stackSentinel.py not found in expected locations')
    print('Available paths checked:')
    for path in possible_paths:
        print(f'  {path} - exists: {path.exists()}')

# Go to work_dir processing directory
os.chdir(isce_run_dir)
print(f'Work directory; {os.getcwd()}')

In [ ]:
#  Add ISCE2 topsStack to PATH and PYTHONPATH for stackSentinel.py
isce2_topsStack_path = Path('/home/roy/miniconda3/envs/isce2/share/isce2/topsStack')
isce2_share_path = Path('/home/roy/miniconda3/envs/isce2/share/isce2')

# Add to PATH for command line execution
os.environ['PATH'] = str(isce2_topsStack_path) + ':' + os.environ['PATH']

# Add to PYTHONPATH for Python module imports
if 'PYTHONPATH' in os.environ:
    os.environ['PYTHONPATH'] = str(isce2_share_path) + ':' + os.environ['PYTHONPATH']
else:
    os.environ['PYTHONPATH'] = str(isce2_share_path)

print(f"✓ stackSentinel.py is now available from: {isce2_topsStack_path}")

# Go to work_dir processing directory
os.chdir(isce_run_dir)
print(f'Work directory; {os.getcwd()}')

In [ ]:
# Generate config and run file for generation of coregistrated SLCs

processing_bound = ' '.join(str(x) for x in aoi) # SNWE
args = f'stackSentinel.py -s {slc_dir} -d {dem_path} -b "{processing_bound}" -a {aux_dir} -o {orbit_dir} -C NESD  -W slc'
args += ' --num_proc4topo 10 --num_proc 10' # to add multi threading
print(args)

# Creates configs and run_files
subprocess.run(args, shell=True, close_fds=True)

# List created run files
run_files = list(run_dir.glob('run_*'))
print(f'Number of run files: {len(run_files)}')
run_files

In [ ]:
# Step 1 - unpack_topo_reference
# Directory: reference and geom_reference
run_file = list(run_dir.glob('run_01*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 2
# Directory: secondarys 
run_file = list(run_dir.glob('run_02*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 3 average baseline
# Directory: baselines
run_file = list(run_dir.glob('run_03*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 4 extract_burst_overlaps
# Directory: reference/overlap
run_file = list(run_dir.glob('run_04*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)

out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 5 overlap_geo2rdr
# Directory: coreg_secondarys
run_file = list(run_dir.glob('run_05*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 6 overlap_resample
# Directory: coreg_secondarys
run_file = list(run_dir.glob('run_06*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 7 pairs_misreg
# Directory: ESD 
run_file = list(run_dir.glob('run_07*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 8 pairs_misreg
# Directory: timeseries_misreg
run_file = list(run_dir.glob('run_08*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 9 pairs_misreg
# Directory: fullBurst_geo2rdr!
run_file = list(run_dir.glob('run_09*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 10 pairs_misreg
# Directory: fullBurst_resample!
run_file = list(run_dir.glob('run_10*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 11 pairs_misreg
# Directory: extract_stack_valid_region'
run_file = list(run_dir.glob('run_11*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 12 pairs_misreg
# Directory: merge_reference_secondary_slc
run_file = list(run_dir.glob('run_12*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 13 pairs_misreg
# Directory: grid_baseline
run_file = list(run_dir.glob('run_13*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

# Generate interferograms


In [ ]:
# Ensure we use the correct path that was found in the previous cell
download_file = stack_path / 'interferogramStack.py'
url = 'https://raw.githubusercontent.com/mgovorcin/isce2_topsStack_ifg_network/main/interferogramStack.py'

# Create the directory if it doesn't exist
download_file.parent.mkdir(parents=True, exist_ok=True)

filename, headers = urllib.request.urlretrieve(url, filename=str(download_file))

download_file.chmod(33791)
subprocess.run(f'interferogramStack.py -h', shell=True)

In [ ]:
ifg_args = dict(
            network = 'full',       # [single_reference, sequential, delaunay, full]
            # num_connections = 2,          # connection number of interferograms between each date for sequential network
            periodic_connections = None,  # number of periodic interferograms in days [180, 365], str or list
            periodic_tolerance = 5,       # tolerance for selection of periodic interferograms around the defined period
            single_reference_date = None, # reference date for single reference network, e.g. 2015-01-23
            start_date = None,            # Start date for interferogram network generation, e.g. 2015-01-23
            end_date = None,              # End date for interferogram network generation, e.g. 2015-01-23
            max_bperp = None,             # Threshold for Maximum Perpendicular baseline [in meters]
            max_btemp = None,             # Threshold for Maximum Temporal baseline [in days]
            azimuth_looks = 3,            # Number of looks in azimuth for interferogram multi-looking
            range_looks = 9,              # Number of looks in range for interferogram multi-looking
            filter_strength = 0.5,        # Filter strength for interferogram filtering
            unw_method = 'snaphu',        # Unwrapping method
            force = True                  # Overwrite run files directory
            )

ifg_cmd_params = dict(
            network = '-n',       
            num_connections = '-c',         
            periodic_connections = '-p',  
            periodic_tolerance = '-pt',       
            single_reference_date = '-sr', 
            start_date = '--start_date',           
            end_date = '--end_date',              
            max_bperp = '--max_bperp',             
            max_btemp = '--max_btemp',            
            azimuth_looks = '-z',           
            range_looks = '-r',             
            filter_strength = '-f',        
            unw_method = '-u',        
            force = '--force'                  
            )

ifg_args

In [ ]:
cmd = f'interferogramStack.py -s {isce_run_dir}'
for arg in ifg_args.keys():
    if ifg_args[arg]:
        if arg == 'force':
            cmd +=  ' ' + ifg_cmd_params[arg]
        else:
            cmd +=  ' ' + ifg_cmd_params[arg] + ' ' + str(ifg_args[arg])

print(cmd)
subprocess.run(cmd, shell=True)
display.Image(f'{isce_run_dir}/interferogram_network.png',width=1000, height=1000)

# List the run files
run_files = list(run_ifg_dir.glob('run_*'))
print(f'Number of run files: {len(run_files)}')
run_files

In [ ]:
# Step 21 
# Directory: generate_burst_igram!
run_file = list(run_ifg_dir.glob('run_21*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 22
# Directory: _merge_burst_igram!
run_file = list(run_ifg_dir.glob('run_22*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 23
# Directory: filter_coherence!
run_file = list(run_ifg_dir.glob('run_23*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

In [ ]:
# Step 24
# Directory: coarse_interferograms
run_file = list(run_ifg_dir.glob('run_24*'))[0]
print(f'RUNNING {run_file}!')
run_file.chmod(33791)
out = subprocess.run(str(run_file), shell=True, check=True, stdout=subprocess.PIPE, close_fds=True)
print(out.stdout.decode("utf-8"))
print('STEP FINISHED - SUCESS!!!' if  out.returncode == 0 else 'STEP FINISHED - FAILED!!!')

# Process more interferograms

In [ ]:
# Lets change network to delaunay
ifg_args['network'] = 'delaunay'

In [ ]:
cmd = f'interferogramStack.py -s {isce_run_dir}'
for arg in ifg_args.keys():
    if ifg_args[arg]:
        if arg == 'force':
            cmd +=  ' ' + ifg_cmd_params[arg]
        else:
            cmd +=  ' ' + ifg_cmd_params[arg] + ' ' + str(ifg_args[arg])

print(cmd)
subprocess.run(cmd, shell=True)

# List the run files
run_files = list(run_ifg_dir.glob('run_*'))
print(f'Number of run files: {len(run_files)}')
run_files

In [ ]:
display.Image(f'{isce_run_dir}/interferogram_network.png',width=500, height=500)